In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense
import logging 
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

In [6]:
def load_coffee_data(): 
    """
        Creates a coffee roasting data set. roasting duration: 12-15 minutes is best 
        temperate range: 170 - 260C is best
    """
    rng = np.random.default_rng(2)
    X = rng.random(400).reshape(-1, 2) 
    X[:, 1] = X[:, 1] * 4 + 11.5 
    X[:, 0] = X[: , 0] * (285 - 150) + 150 
    Y = np.zeros(len(X))

    i = 0
    for t, d in X: 
        y = -3/(260 - 175)*t + 21 
        if (t > 175 and t < 260 and d > 12 and d < 15 and d <= y): 
            Y[i] = 1 
        else:  
            Y[i] = 0
        i += 1

    return (X, Y.reshape(-1, 1))

In [7]:
X, Y = load_coffee_data()
print(X.shape, Y.shape)

(200, 2) (200, 1)


Let's plot the coffee roasting data below. The two features are Temperature in Celsius and Duration in minutes. [Coffee Roasting at Home](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) suggeests that the duration is best kept between 12 and 15 minutes while the temp should be between 175 and 260 degrees Celsius. Of course, as temperature rises, the duration should shrink.

<center> <img  src="coffee_roasting.png" width="400" /> <center/> 

#### Normalize Data 

Fitting the weights to the data will proceed more quickly if the data is normalized. This is the same procedure you used in Course 1 where features are each normalized to have a similar range. The procedure below uses a Keras [normalization layer](https://keras.io/api/layers/preprocessing_layers/numerical/normalization/) . It has the following steps: 
* create a "Normalization Layer". Note, as applied here, this is not a layer in your model.
* 'adapt' the data. This learns the mean and variance of the data set and saves the values internally.
* normalize the data.
It is important to apply normalization to any future data that utilizes the learned model.

In [40]:
print(f"Temperature max, min pre normalization: {np.max(X[:, 0]):0.2f}, {np.min(X[:, 0]):0.2f}") 
print(f"Duration    max, min pre normalization: {np.max(X[:, 1]):0.2f}, {np.min(X[:, 1]):0.2f}") 

norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)
Xn = norm_l(X)

print(f"Temperature max, min post normalization: {np.max(Xn[:, 0]):0.2f}, {np.min(Xn[:, 0]):0.2f}") 
print(f"Duration    max, min post normalization: {np.max(Xn[:, 1]):0.2f}, {np.min(Xn[:, 1]):0.2f}") 

Temperature max, min pre normalization: 284.99, 151.32
Duration    max, min pre normalization: 15.45, 11.51
Temperature max, min post normalization: 1.66, -1.69
Duration    max, min post normalization: 1.79, -1.70


Tile/copy our data to increase the training set size and reduce the number of training epochs

In [42]:
Xt = np.tile(Xn, (1000, 1)) 
Yt = np.tile(Y, (1000, 1)) 
print(Xt.shape, Yt.shape)

(200000, 2) (200000, 1)


## Tensorflow Model

<center> <img  src="C2_W1_RoastingNetwork.PNG" width="400" /> <center/> 

In [32]:
tf.random.set_seed(1234) # applied to achieve consistent results 
model = Sequential (
    [
        tf.keras.Input(shape=(2, )),
        Dense(3, activation = 'sigmoid', name = 'layer1'), 
        Dense(1, activation= 'sigmoid', name = 'layer2')
    ]
)

>**Note 1**: The `tf.keras.Input(shape=(2, ))` specifies the expected shape of the input. This allows Tensorflow to size the weight and bias parameters at this point. This is useful when exploring Tensorflow models. This statement can be omitted in practice and Tensorflow will size the network parameters when the input data is specified in the `model fit` statement.
>**Note 2**: Including the sigmoid activation in the final layer is not considerred best practice. It would instead be accounted for in the loss which improves numerical stability. This will be described in more details in a later lab.

In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                  │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13 (52.00 B)

 Trainable params: 13 (52.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
L1_num_params = 2 * 3 + 3 # W1 parameters + b1 parameters 
L2_num_params = 3 * 1 + 1 # W2 parameters + b2 parameters
print(f"L1 params = {L1_num_params}")
print(f"L2 params = {L2_num_params}") 

L1 params = 9
L2 params = 4


Let's examine the weights and biases Tensorflow has instantiated. The weight $W$ should be size (number of features in input, number of units in the layer) while the bias b size should match the number of units in the layer: 
* In the first layer with 3 units, we expect W to have a size of (2, 3) and b should have 3 elements.
* In the second layer with 1 unit, we expect W to have a size of (3, 1) and b should have 1 element.

In [39]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb1{b2.shape}:", b2)

W1(2, 3):
 [[-1.0711559  -0.43849158 -0.536871  ]
 [-0.4239055   0.44875026 -0.10167551]] 
b1(3,): [0. 0. 0.]
W2(3, 1):
 [[-0.11579835]
 [ 0.8491305 ]
 [ 0.99101317]] 
b1(1,): [0.]


The following statements will be described in detail in Week 2. For now: 
* The `model.compile` statement defines a loss and specifies a compile optimization
* The `model.fit` statement runs gradient descent and fits the weights to the data

In [46]:
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(), 
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
) 

model.fit (
    Xt, Yt, 
    epochs = 10,
) 

Epoch 1/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 185us/step - loss: 1.3581e-04
Epoch 2/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 184us/step - loss: 1.1423e-04
Epoch 3/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 184us/step - loss: 9.6827e-05
Epoch 4/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 189us/step - loss: 8.7922e-05
Epoch 5/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 184us/step - loss: 7.3495e-05
Epoch 6/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 184us/step - loss: 6.8123e-05
Epoch 7/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 187us/step - loss: 5.4648e-05
Epoch 8/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 184us/step - loss: 2.1071e-04
Epoch 9/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 184us/step - loss: 4.3753e-05
Epoch 10/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 183us/step - loss: 3.8415e-05


#### Epochs and batches 

In the `fit` statement above, the number of `epochs` was set to 10. This specifies that the entire data set should be applied during training 10 times. During training, you see output describing the progress of training like this: 

```
Epoch 1/10
6250/6250 [==============================] - 6s 910us/step - loss: 0.1782
```
The first line, `Epoch 1/ 10`, describes which epoch the model is currently running. For efficiency, the training data is broken into 'batches'. The default size of a batch in Tensorflow is 32. There are 200000 examples in our expanded data set or 6250 batches.

#### Updated Weights 

In [48]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

W1:
 [[ 17.966278   -12.427142     0.27662078]
 [ 14.914531    -0.32282764  11.793835  ]] 
b1: [  2.6942885 -13.056967   14.072244 ]
W2:
 [[ -90.9689  ]
 [-102.874344]
 [  93.43234 ]] 
b2: [-26.530548]


For the purpose of the next discussion, instead of using the weights you got right away, you will first set some weights we saved from a previous training run. This is so that this notebook remains robust to changes in Tensorflow over time. Different training runs can produce somewhat different results and the following discussion applies when the model has the weights you will load below. 

In [54]:
W1 = np.array([
    [ 17.966278,   -12.427142,     0.27662078],
    [ 14.914531,    -0.32282764,  11.793835  ]]  )
b1 = np.array([  2.6942885, -13.056967,   14.072244 ])
W2 = np.array( [[ -90.9689  ],
 [-102.874344],
 [  93.43234 ]])
b2 = np.array([-26.530548])

# Replace the weights from your trained model with
# the values above.
model.get_layer("layer1").set_weights([W1,b1])
model.get_layer("layer2").set_weights([W2,b2])

In [55]:
# Check if the weights are successfully replaced
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

W1:
 [[ 17.966278   -12.427142     0.27662078]
 [ 14.914531    -0.32282764  11.793835  ]] 
b1: [  2.6942885 -13.056967   14.072244 ]
W2:
 [[ -90.9689  ]
 [-102.874344]
 [  93.43234 ]] 
b2: [-26.530548]


#### Predictions

Once you have trained model, you can use it to make predictions. Recall that the output of the our model is probability. In this case, the probability of a good roast. To make a decision, one must apple the probability to a threshold. 

Let's start by creating input data. The model is expecting one or more examples where examples are in the rows of matrix. In this case, we have two features so the matrix will be (m, 2) where m is the number of examples. Recall, we have normalized the input features so we must normalize our test data as well.

In [63]:
X_test = np.array([
     [200, 13.9],
     [200, 17]])
X_testn = norm_l(X_test)
predictions = model.predict(X_testn)
print("predictions = \n", predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
predictions = 
 [[9.9995965e-01]
 [3.4386105e-11]]


In [61]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)): 
    if predictions[i] >= 0.5: 
        yhat[i] = 1 
    else: 
        yhat[i] = 0
print(f"decisions = \n{yhat}")

decisions = 
[[1.]
 [0.]]


This can be accomplished more succinctly:

In [62]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

decisions = 
[[1]
 [0]]
